# Merging StatsBomb Events with SkillCorner Tracking Data

This notebook demonstrates the **mapping-based merge approach** similar to the StatsBomb-Hudl guide.

## Approach
1. Load the match mapping file (links SkillCorner match IDs ↔ StatsBomb match IDs)
2. Select one match to work with
3. **Add SkillCorner IDs to StatsBomb events** using `pd.merge(..., how='left')`
4. **Add StatsBomb IDs to SkillCorner tracking** using `pd.merge(..., how='left')`
5. Now both datasets are ready to be joined whenever needed!

This gives you maximum flexibility to work with raw data directly.

## Step 1: Import Libraries and Load Mapping

In [1]:
import pandas as pd
import json
from pathlib import Path
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load the match mapping file
mapping_df = pd.read_csv('data/mapping_ids/skc_sb_match_mapping.csv')

print(f"Loaded {len(mapping_df)} match mappings")
print(f"\nColumns: {mapping_df.columns.tolist()}")
print(f"\nFirst few mappings:")
mapping_df.head()

Loaded 380 match mappings

Columns: ['skc_match_id', 'sb_match_id', 'sb_home_team_id', 'date', 'home_team', 'away_team']

First few mappings:


,skc_match_id,sb_match_id,sb_home_team_id,date,home_team,away_team
0,1410827,3925227,1884,2024-02-23,Nagoya Grampus,Kashima Antlers
1,1410828,3925226,1889,2024-02-23,Sanfrecce Hiroshima,Urawa Red Diamonds
2,1411836,3925231,1895,2024-02-24,Shonan Bellmare,Kawasaki Frontale
3,1411839,3925233,1894,2024-02-24,Sagan Tosu,Albirex Niigata
4,1411840,3925230,4652,2024-02-24,Jubilo Iwata,Vissel Kobe


## Step 2: Select First Match

In [3]:
# Get the first match from mapping
first_match = mapping_df.iloc[0]

skc_match_id = first_match['skc_match_id']
sb_match_id = first_match['sb_match_id']
sb_home_team_id = first_match['sb_home_team_id']

print(f"="*70)
print(f"SELECTED MATCH")
print(f"="*70)
print(f"SkillCorner Match ID: {skc_match_id}")
print(f"StatsBomb Match ID:   {sb_match_id}")
print(f"Date:                 {first_match['date']}")
print(f"Home Team:            {first_match['home_team']}")
print(f"Away Team:            {first_match['away_team']}")
print(f"="*70)

SELECTED MATCH
SkillCorner Match ID: 1410827
StatsBomb Match ID:   3925227
Date:                 2024-02-23
Home Team:            Nagoya Grampus
Away Team:            Kashima Antlers


## Step 3: Load StatsBomb Events and Add SkillCorner IDs

This follows the **exact same pattern** as adding Wyscout IDs to StatsBomb data in the Hudl guide.

In [4]:
# Load all StatsBomb events
print("Loading StatsBomb events...")
with open('data/sb_data/sb_events.json', 'r', encoding='utf-8') as f:
    all_sb_events = json.load(f)

print(f"Total events in file: {len(all_sb_events):,}")

# Convert to DataFrame
events_df = pd.json_normalize(all_sb_events)

print(f"\nStatsBomb events DataFrame shape: {events_df.shape}")
print(f"Columns: {len(events_df.columns)}")

Loading StatsBomb events...
Total events in file: 1,244,341

StatsBomb events DataFrame shape: (1244341, 162)
Columns: 162


In [5]:
# Add SkillCorner match IDs to StatsBomb events using LEFT JOIN
# This is the same as: events_df = pd.merge(events_df, mapping, on='match_id', how='left')

print("Adding SkillCorner match IDs to StatsBomb events...")
print(f"\nBefore merge:")
print(f"  Columns: {len(events_df.columns)}")
print(f"  Has 'skc_match_id' column: {'skc_match_id' in events_df.columns}")

# Merge using left join (equivalent to the Hudl guide approach)
events_df = pd.merge(
    events_df,
    mapping_df[['sb_match_id', 'skc_match_id', 'date', 'home_team', 'away_team']],
    left_on='match_id',  # StatsBomb's match ID column
    right_on='sb_match_id',
    how='left'
)

print(f"\nAfter merge:")
print(f"  Columns: {len(events_df.columns)}")
print(f"  Has 'skc_match_id' column: {'skc_match_id' in events_df.columns}")
print(f"  Events with SkillCorner ID: {events_df['skc_match_id'].notna().sum():,}/{len(events_df):,}")

print(f"\n✓ StatsBomb events now have SkillCorner match IDs!")

Adding SkillCorner match IDs to StatsBomb events...

Before merge:
  Columns: 162
  Has 'skc_match_id' column: False

After merge:
  Columns: 167
  Has 'skc_match_id' column: True
  Events with SkillCorner ID: 1,244,341/1,244,341

✓ StatsBomb events now have SkillCorner match IDs!


In [6]:
# Filter events for our selected match
match_events_df = events_df[events_df['match_id'] == sb_match_id].copy()

print(f"="*70)
print(f"STATSBOMB EVENTS FOR SELECTED MATCH")
print(f"="*70)
print(f"Total events: {len(match_events_df)}")
print(f"\nKey columns with cross-reference IDs:")
print(match_events_df[['id', 'match_id', 'skc_match_id', 'type.name', 'team.name', 'player.name']].head(10))

STATSBOMB EVENTS FOR SELECTED MATCH
Total events: 3639

Key columns with cross-reference IDs:
                                           id  match_id  skc_match_id  \
1237007  73c68d7a-9dc1-4a78-8d10-450b64a9e796   3925227       1410827   
1237008  0671a469-a118-4ecb-a119-8948daa0be7d   3925227       1410827   
1237009  9e30ad03-f65f-48f4-ba3a-ba574ad9a737   3925227       1410827   
1237010  dd6cd9c2-b626-4a08-8238-7cc5ce73f904   3925227       1410827   
1237011  028e9a97-663a-4785-9fd9-5c25708a401e   3925227       1410827   
1237012  51b871e9-20af-46db-8b3f-fc9d50b91f89   3925227       1410827   
1237013  3330c1d8-295b-4f19-b0e4-0d9c15988859   3925227       1410827   
1237014  623fe627-08a1-4688-a6d6-1e26e38687bf   3925227       1410827   
1237015  c215354d-611e-4abb-9201-47df91c4db65   3925227       1410827   
1237016  a567e117-4ab1-4aae-a0b0-73ba72317874   3925227       1410827   

             type.name        team.name    player.name  
1237007    Starting XI   Nagoya Grampus      

## Step 4: Load SkillCorner Tracking Data and Add StatsBomb IDs

Now we do the reverse - add StatsBomb IDs to SkillCorner tracking data.

In [ ]:
# Load SkillCorner tracking data for the selected match
tracking_path = Path(f'data/tracking_j1_2024/{skc_match_id}_tracking_extrapolated.jsonl')

print(f"Loading SkillCorner tracking data from: {tracking_path}")

if tracking_path.exists():
    # JSONL format - one JSON object per line
    tracking_frames = []
    with open(tracking_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()  # Remove whitespace
            if not line:  # Skip empty lines
                continue
            try:
                tracking_frames.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Warning: Skipping line {line_num} due to JSON error: {e}")
                continue
    
    print(f"\n✓ Loaded {len(tracking_frames)} tracking frames")
    
    # Convert to DataFrame
    tracking_df = pd.DataFrame(tracking_frames)
    
    print(f"\nTracking DataFrame shape: {tracking_df.shape}")
    print(f"Columns: {tracking_df.columns.tolist()[:10]}...")  # Show first 10 columns
else:
    print(f"\n✗ Tracking file not found: {tracking_path}")
    tracking_df = None

Loading SkillCorner tracking data from: data\tracking_j1_2024\1410827_tracking_extrapolated.jsonl


JSONDecodeError: Expecting value: line 2 column 1 (char 2)

In [ ]:
if tracking_df is not None:
    # Add StatsBomb match ID to tracking data using LEFT JOIN
    print("Adding StatsBomb match IDs to SkillCorner tracking...")
    print(f"\nBefore merge:")
    print(f"  Columns: {len(tracking_df.columns)}")
    print(f"  Has 'sb_match_id' column: {'sb_match_id' in tracking_df.columns}")
    
    # Check if tracking data has a match_id field
    if 'match_id' in tracking_df.columns:
        print(f"  Tracking match_id values: {tracking_df['match_id'].unique()}")
    
    # Merge - add StatsBomb IDs to tracking (opposite direction from events)
    # Since tracking data has frames, not match-level records, we'll add match metadata
    tracking_df['skc_match_id_original'] = skc_match_id  # Add the SkillCorner ID
    
    tracking_df = pd.merge(
        tracking_df,
        mapping_df[['skc_match_id', 'sb_match_id', 'date', 'home_team', 'away_team']],
        left_on='skc_match_id_original',
        right_on='skc_match_id',
        how='left'
    )
    
    print(f"\nAfter merge:")
    print(f"  Columns: {len(tracking_df.columns)}")
    print(f"  Has 'sb_match_id' column: {'sb_match_id' in tracking_df.columns}")
    print(f"  Frames with StatsBomb ID: {tracking_df['sb_match_id'].notna().sum():,}/{len(tracking_df):,}")
    
    print(f"\n✓ SkillCorner tracking now has StatsBomb match IDs!")
else:
    print("\nSkipping tracking merge (no tracking data loaded)")

In [ ]:
if tracking_df is not None:
    print(f"="*70)
    print(f"SKILLCORNER TRACKING FOR SELECTED MATCH")
    print(f"="*70)
    print(f"Total frames: {len(tracking_df)}")
    print(f"\nSample tracking data with cross-reference IDs:")
    
    # Show key columns
    display_cols = [col for col in ['timestamp', 'period', 'skc_match_id', 'sb_match_id', 'home_team', 'away_team'] 
                    if col in tracking_df.columns]
    
    if display_cols:
        print(tracking_df[display_cols].head(10))
    else:
        print("\nAvailable columns:")
        print(tracking_df.columns.tolist())
        print("\nFirst few rows:")
        print(tracking_df.head(3))

## Step 5: Demonstrate Merging Events with Tracking

Now that both datasets have cross-reference IDs, we can merge them based on our analysis needs.

In [ ]:
if tracking_df is not None:
    print(f"="*70)
    print(f"READY TO MERGE!")
    print(f"="*70)
    
    print(f"\nDataset Summary:")
    print(f"\n1. StatsBomb Events:")
    print(f"   - Total events: {len(match_events_df):,}")
    print(f"   - Has skc_match_id: ✓")
    print(f"   - Match ID: {sb_match_id}")
    
    print(f"\n2. SkillCorner Tracking:")
    print(f"   - Total frames: {len(tracking_df):,}")
    print(f"   - Has sb_match_id: ✓")
    print(f"   - Match ID: {skc_match_id}")
    
    print(f"\n3. Next Steps:")
    print(f"   You can now merge these datasets based on:")
    print(f"   - Timestamp (temporal join)")
    print(f"   - Event type + time window")
    print(f"   - Specific analysis needs")
    
    print(f"\n4. Example Merge Approaches:")
    print(f"   A. Nearest timestamp: pd.merge_asof(events, tracking, on='timestamp')")
    print(f"   B. Time window: Find tracking frames within ±N seconds of each event")
    print(f"   C. Specific events: Filter events (e.g., passes) then join with tracking")
else:
    print("\nCannot demonstrate merge - tracking data not loaded")

## Step 6: Example - Temporal Join (Nearest Timestamp)

Let's demonstrate one merge approach: joining events with their nearest tracking frame.

In [ ]:
if tracking_df is not None and 'timestamp' in match_events_df.columns and 'timestamp' in tracking_df.columns:
    print("Attempting temporal merge (nearest timestamp)...\n")
    
    # Prepare data for merge_asof (requires sorted data)
    events_sorted = match_events_df.sort_values('timestamp').copy()
    tracking_sorted = tracking_df.sort_values('timestamp').copy()
    
    # Select key event columns to avoid column explosion
    event_cols = ['id', 'timestamp', 'type.name', 'team.name', 'player.name', 'location']
    event_cols = [c for c in event_cols if c in events_sorted.columns]
    
    # Perform merge_asof (finds nearest tracking frame for each event)
    merged = pd.merge_asof(
        events_sorted[event_cols],
        tracking_sorted,
        on='timestamp',
        direction='nearest',
        suffixes=('_event', '_tracking')
    )
    
    print(f"✓ Merged {len(merged)} events with nearest tracking frames")
    print(f"\nMerged data shape: {merged.shape}")
    print(f"\nSample merged data:")
    print(merged[['id', 'timestamp', 'type.name', 'team.name']].head(10))
    
    print(f"\n✓ Success! Events are now linked with tracking data.")
    
elif tracking_df is not None:
    print("Cannot perform temporal merge - timestamp column missing")
    print(f"\nAvailable columns in events: {match_events_df.columns.tolist()[:10]}...")
    print(f"Available columns in tracking: {tracking_df.columns.tolist()[:10]}...")
else:
    print("Cannot perform merge - tracking data not loaded")

## Summary

### What We Accomplished

Following the **StatsBomb-Hudl merge pattern**, we:

1. ✓ Loaded match mapping file (SkillCorner ↔ StatsBomb)
2. ✓ Selected one match to work with
3. ✓ Added SkillCorner IDs to StatsBomb events using `pd.merge(..., how='left')`
4. ✓ Added StatsBomb IDs to SkillCorner tracking using `pd.merge(..., how='left')`
5. ✓ Demonstrated temporal merge using `pd.merge_asof()`

### Key Advantages of This Approach

- **Same pattern as Hudl guide**: Uses familiar pd.merge() approach
- **Maximum flexibility**: Work with raw data formats directly
- **No black box**: You control exactly how data is joined
- **Scalable**: Apply same pattern to all 376 successfully mapped matches

### Comparison with SkillCorner Toolkit

| Aspect | Toolkit Approach | Mapping Approach |
|--------|-----------------|------------------|
| Output | freeze_frame_format.json | Flexible (your choice) |
| Control | Pre-packaged | Full control |
| Speed | Automated batch | Manual (but flexible) |
| Use case | Quick analysis | Custom research |

**Both approaches are valid!** Use the toolkit for quick results, use mapping for custom analysis.